In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

_NOTEBOOK_START = Path.cwd().resolve()
for candidate in (_NOTEBOOK_START, *_NOTEBOOK_START.parents):
    if (candidate / "requirements.txt").is_file() and (candidate / "grid_transform").is_dir():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError(f"Could not locate repo root from {_NOTEBOOK_START}")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from grid_transform.notebook_bootstrap import bootstrap_notebook

globals().update(bootstrap_notebook(REPO_ROOT))
print(f"Notebook bootstrap ready: {PROJECT_DIR}")


In [ ]:

# ── Paths ─────────────────────────────────────────────────────────────────────
VTLN_DIR    = str(PROJECT_DIR / "VTLN")

BASE         = str(PROJECT_DIR / "vocal-tract-seg")
DATA_DIR_A   = f"{BASE}/data_80/2008-003^01-1791/test"
CONTOURS_A   = f"{BASE}/results/nnunet_080/inference_contours/2008-003^01-1791/test"
FRAME_A      = 143020

# ── Load Speaker A (nnUNet) ───────────────────────────────────────────────────
print("=" * 60)
print("Loading Speaker A  (nnUNet)")
print("=" * 60)
image_A, contours_A = load_frame_npy(FRAME_A, DATA_DIR_A, CONTOURS_A)

# ── Load all VTLN speakers ────────────────────────────────────────────────────
vtln_names = discover_vtln_speakers(VTLN_DIR)
print(f"\n{'=' * 60}")
print(f"VTLN speakers found: {len(vtln_names)}")
print("=" * 60)

speakers = {}   # key: name string -> {'image': PIL, 'contours': dict}
speakers["nnUNet_A"] = {"image": image_A, "contours": contours_A, "source": "nnUNet"}

for name in vtln_names:
    try:
        img, ctr = load_frame_vtln(name, VTLN_DIR)
        speakers[name] = {"image": img, "contours": ctr, "source": "VTLN"}
    except Exception as e:
        print(f"  WARNING: could not load {name}: {e}")

print(f"\nTotal speakers loaded: {len(speakers)}")
for k, v in speakers.items():
    print(f"  {k:30s}  [{v['source']}]  {len(v['contours'])} contours")


In [ ]:

# Build grids for all speakers 
print("Building grids …")
grids = {}
for name, spk in speakers.items():
    frame_num = FRAME_A if name == "nnUNet_A" else 0
    try:
        g = build_grid(spk["image"], spk["contours"],
                       n_vert=9, n_points=250, frame_number=frame_num)
        grids[name] = g
        print(f"  ✓  {name}")
    except Exception as e:
        print(f"  ✗  {name}: {e}")

speaker_keys = list(grids.keys())
n = len(speaker_keys)
print(f"\nGrids built: {n}")


In [ ]:

# Figure 1: RAW images  (ALL speakers, no annotations)
all_keys = list(speakers.keys())          # all 10 (including failed grids)
n_all  = len(all_keys)
ncols = min(n_all, 5)
nrows = (n_all + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 5 * nrows))
axes = np.array(axes).reshape(-1)

for i, key in enumerate(all_keys):
    ax = axes[i]
    ax.imshow(speakers[key]["image"], cmap="gray")
    failed = key not in grids
    tag = "  ⚠ no grid" if failed else ""
    ax.set_title(f"{key}{tag}", fontsize=9, fontweight="bold",
                 color="red" if failed else "black")
    ax.axis("off")

for j in range(n_all, len(axes)):
    axes[j].axis("off")

fig.suptitle(f"All {n_all} speakers — raw images (no annotations)",
             fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


In [ ]:

# Figure 2: images WITH contour annotations (+ grid where available)
all_keys = list(speakers.keys())          # all 10
n_all  = len(all_keys)
ncols = min(n_all, 5)
nrows = (n_all + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 6 * nrows))
axes = np.array(axes).reshape(-1)

cmap_art = plt.get_cmap("tab10")

for i, key in enumerate(all_keys):
    ax   = axes[i]
    spk  = speakers[key]
    g    = grids.get(key)            # None for speakers without a grid

    # --- raw image ---
    ax.imshow(spk["image"], cmap="gray")

    # --- articulatory contours (always shown) ---
    for ci, (lbl, pts) in enumerate(spk["contours"].items()):
        c = cmap_art(ci % 10)
        ax.plot(pts[:, 0], pts[:, 1], "-", color=c, lw=1.5, alpha=0.85,
                label=lbl)

    if g is not None:
        # --- grid lines ---
        for h in g.horiz_lines:
            ax.plot(h[:, 0], h[:, 1], color="cyan",   lw=0.8, alpha=0.6)
        for v in g.vert_lines:
            ax.plot(v[:, 0], v[:, 1], color="yellow",  lw=0.8, alpha=0.6)

        # --- cervical-spine centres ---
        cc = g.cervical_centers or {}
        for clbl, cpt in cc.items():
            ax.plot(*cpt, "rs", ms=6, mec="white", mew=1)
            ax.annotate(clbl.upper(), cpt, fontsize=7, color="red",
                        xytext=(3, -9), textcoords="offset points")

        # --- M1 / P1 ---
        if g.M1 is not None:
            ax.plot(*g.M1, "g^", ms=8, mec="white", mew=1, label="M1")
        if hasattr(g, "P1_point") and g.P1_point is not None:
            ax.plot(*g.P1_point, "mp", ms=9, mec="white", mew=1, label="P1")

    # --- legend (first panel only) ---
    if i == 0:
        ax.legend(fontsize=6, loc="upper right", ncol=2,
                  framealpha=0.5, handlelength=1)

    has_grid = g is not None
    src_tag  = f"[{spk['source']}]"
    grid_tag = "" if has_grid else "  (contours only)"
    ax.set_title(f"{key}\n{src_tag}{grid_tag}", fontsize=9, fontweight="bold",
                 color="black" if has_grid else "darkorange")

    h_img, w_img = np.asarray(spk["image"]).shape[:2]
    ax.set_xlim(0, w_img); ax.set_ylim(h_img, 0)
    ax.axis("off")

for j in range(n_all, len(axes)):
    axes[j].axis("off")

fig.suptitle("All speakers — contour annotations (+ grid where available)",
             fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


In [ ]:

# ── Inspect contour labels across speakers (summary) ─────────────────────────
for key, spk in speakers.items():
    labels = sorted(spk["contours"].keys())
    print(f"{key:30s}  [{spk['source']:6s}]  {len(labels)} labels: {labels[:5]}{'...' if len(labels)>5 else ''}")

# Find common labels across ALL speakers
all_label_sets = [set(spk["contours"].keys()) for spk in speakers.values()]
common_labels_all = sorted(set.intersection(*all_label_sets))
print(f"\nCommon labels across ALL {len(speakers)} speakers: {len(common_labels_all)}")
if common_labels_all:
    print(f"  {common_labels_all}")

# Find common labels across VTLN speakers only
vtln_label_sets = [set(spk["contours"].keys()) for k, spk in speakers.items() if spk["source"] == "VTLN"]
common_labels_vtln = sorted(set.intersection(*vtln_label_sets)) if vtln_label_sets else []
print(f"\nCommon labels across {len(vtln_label_sets)} VTLN speakers: {len(common_labels_vtln)}")
if common_labels_vtln:
    print(f"  {common_labels_vtln}")
    # Show point counts for these labels
    for lbl in common_labels_vtln:
        ncs = {k: spk["contours"][lbl].shape[0] for k, spk in speakers.items() if lbl in spk["contours"]}
        print(f"    {lbl:25s}  nc per speaker: {list(ncs.values())}")


In [ ]:

# =====================================================================
#  MEAN-SPEAKER  —  Helper functions
# =====================================================================

def flatten_pts(pts):
    """(nc, 2) -> (2*nc,) flattened as [x1,y1,x2,y2,...]."""
    return np.asarray(pts, dtype=float).reshape(-1)


def unflatten_pts(x):
    """(2*nc,) -> (nc, 2)."""
    x = np.asarray(x, dtype=float)
    assert x.ndim == 1 and x.size % 2 == 0, f"Expected flat vector with even length, got {x.shape}"
    return x.reshape(-1, 2)


def resample_polyline(pts, n_out):
    """
    Resample an ordered 2-D polyline to exactly n_out equi-arclength points.
    Uses linear interpolation along cumulative arc-length.

    Parameters
    ----------
    pts   : (M, 2)  input polyline
    n_out : int      desired number of output points

    Returns
    -------
    resampled : (n_out, 2)
    """
    pts = np.asarray(pts, dtype=float)
    if len(pts) < 2:
        return np.tile(pts[0], (n_out, 1))
    diffs = np.diff(pts, axis=0)
    seg_len = np.linalg.norm(diffs, axis=1)
    arc = np.concatenate([[0.0], np.cumsum(seg_len)])
    total = arc[-1]
    if total < 1e-12:                         # degenerate (all same point)
        return np.tile(pts[0], (n_out, 1))
    arc_norm = arc / total
    t_new = np.linspace(0.0, 1.0, n_out)
    resampled = np.column_stack([
        np.interp(t_new, arc_norm, pts[:, 0]),
        np.interp(t_new, arc_norm, pts[:, 1]),
    ])
    return resampled


def estimate_similarity_umeyama(src, dst):
    """
    Estimate 2-D similarity transform  dst ≈ c * src @ R^T + t
    using the svd method (SVD-based).

    Parameters
    ----------
    src, dst : (nc, 2)  matched point sets

    Returns
    -------
    c : float       uniform scale
    R : (2, 2)      rotation matrix
    t : (2,)        translation vector
    """
    src = np.asarray(src, dtype=float)
    dst = np.asarray(dst, dtype=float)
    n = len(src)
    assert n == len(dst) and n >= 2

    mu_s = src.mean(axis=0)
    mu_d = dst.mean(axis=0)
    S = src - mu_s
    D = dst - mu_d

    var_s = np.sum(S ** 2) / n            # total variance of src

    # Cross-covariance
    Sigma = (D.T @ S) / n                 # (2, 2)

    U, sv, Vt = np.linalg.svd(Sigma)

    # Ensure proper rotation (det = +1)
    det_sign = np.linalg.det(U) * np.linalg.det(Vt)
    diag = np.array([1.0, 1.0 if det_sign > 0 else -1.0])

    R = U @ np.diag(diag) @ Vt            # (2, 2) rotation

    c = float(np.sum(sv * diag) / var_s) if var_s > 1e-12 else 1.0

    t = mu_d - c * (R @ mu_s)             # (2,)

    return c, R, t


def apply_similarity(pts, c, R, t):
    """
    Apply similarity transform:  pts_out = c * pts @ R^T + t

    Parameters
    ----------
    pts : (nc, 2)  or  (2,)
    c   : float
    R   : (2, 2)
    t   : (2,)

    Returns
    -------
    pts_out : same shape as input
    """
    pts = np.asarray(pts, dtype=float)
    single = (pts.ndim == 1)
    if single:
        pts = pts.reshape(1, 2)
    out = c * (pts @ R.T) + t
    return out[0] if single else out


print("Helper functions defined ✓")
print("  flatten_pts, unflatten_pts, resample_polyline")
print("  estimate_similarity_umeyama, apply_similarity")


In [ ]:

# =====================================================================
#  MEAN-SPEAKER  —  Main GPA function  (v2 — pixel-scale preserving)
# =====================================================================

def compute_mean_speaker(contours_by_speaker, nc_resample=50,
                         max_iter=100, tol=1e-6, verbose=True):
    """
    Compute a "mean speaker" via iterative Generalized Procrustes Analysis.

    The template is kept in original pixel coordinates (average speaker
    scale).  Scale collapse is prevented by constraining the arithmetic
    mean of per-speaker scales to 1 after each iteration.

    Returns
    -------
    result : dict  (see keys below)
    """
    speaker_ids = sorted(contours_by_speaker.keys())
    N = len(speaker_ids)
    assert N >= 2, "Need at least 2 speakers"

    # ── 1. Find common articulation labels ────────────────────────────────
    label_sets = [set(contours_by_speaker[s].keys()) for s in speaker_ids]
    common_labels = sorted(set.intersection(*label_sets))
    M = len(common_labels)
    assert M >= 1, "No common labels found across speakers"
    if verbose:
        print(f"Speakers: {N},  Common articulations: {M}")
        print(f"  Labels: {common_labels}")

    # ── 2. Resample all contours to common nc ─────────────────────────────
    nc = nc_resample
    nc_total = M * nc

    contours_resampled = {}
    for s in speaker_ids:
        contours_resampled[s] = {}
        for lbl in common_labels:
            raw = np.asarray(contours_by_speaker[s][lbl], dtype=float)
            if raw.ndim == 1:
                raw = unflatten_pts(raw)
            contours_resampled[s][lbl] = resample_polyline(raw, nc)

    # ── 3. Build per-speaker super-shape (concatenate all articulators) ───
    speaker_means = {}
    for s in speaker_ids:
        parts = [contours_resampled[s][lbl] for lbl in common_labels]
        speaker_means[s] = np.vstack(parts)       # (nc_total, 2)

    if verbose:
        print(f"  Super-shape: ({nc_total}, 2) = {M} labels × {nc} pts")

    # ── 4. GPA — iterate Procrustes alignment ────────────────────────────
    #
    # Key: template stays in PIXEL SCALE.
    #   - Initialise template as the plain mean of all speaker super-shapes.
    #   - Each iteration: align every speaker to the template with a
    #     RIGID transform (rotation + translation, NO scale).
    #     → This guarantees shapes keep their original pixel size
    #       and anatomical proportions.
    #   - Update template = mean of aligned shapes; re-centre.
    #   - Convergence when template stops changing.
    #
    # After convergence we estimate a final SIMILARITY transform
    # (with scale) per speaker so we can report it, but the template
    # itself was built with rigid alignment to preserve shape.

    template = np.mean([speaker_means[s] for s in speaker_ids], axis=0)
    template -= template.mean(axis=0)

    transforms_rigid = {}           # used during iteration

    for iteration in range(1, max_iter + 1):
        aligned = {}
        for s in speaker_ids:
            # Rigid alignment (no scale) — Procrustes with c forced to 1
            c_full, R, t = estimate_similarity_umeyama(speaker_means[s], template)
            # Force rigid: recompute translation without scale
            mu_s = speaker_means[s].mean(axis=0)
            mu_t = template.mean(axis=0)
            t_rigid = mu_t - R @ mu_s              # no scale factor
            transforms_rigid[s] = (R, t_rigid)
            aligned[s] = (speaker_means[s] @ R.T) + t_rigid

        # Update template
        new_template = np.mean([aligned[s] for s in speaker_ids], axis=0)
        new_template -= new_template.mean(axis=0)

        delta = np.sqrt(np.mean((new_template - template) ** 2))
        template = new_template

        if verbose and (iteration <= 5 or iteration % 10 == 0):
            print(f"  iter {iteration:3d}  delta = {delta:.2e}")

        if delta < tol:
            if verbose:
                print(f"  Converged at iter {iteration}  "
                      f"(delta={delta:.2e} < tol={tol:.1e})")
            break
    else:
        if verbose:
            print(f"  Reached max_iter={max_iter}  (delta={delta:.2e})")

    # ── 5. Final similarity transforms (with scale) for reporting ─────────
    transforms = {}
    for s in speaker_ids:
        c, R, t = estimate_similarity_umeyama(speaker_means[s], template)
        transforms[s] = (c, R, t)

    # ── 6. Aligned speaker means & per-label contours ─────────────────────
    speaker_means_aligned = {}
    all_contours_aligned = {}
    for s in speaker_ids:
        R, t_r = transforms_rigid[s]
        speaker_means_aligned[s] = (speaker_means[s] @ R.T) + t_r
        all_contours_aligned[s] = {}
        for lbl in common_labels:
            pts = contours_resampled[s][lbl]
            all_contours_aligned[s][lbl] = (pts @ R.T) + t_r

    # ── 7. Per-label template (for convenience) ───────────────────────────
    template_per_label = {}
    for li, lbl in enumerate(common_labels):
        template_per_label[lbl] = template[li * nc : (li + 1) * nc]

    result = {
        "template_mean":         template,
        "template_per_label":    template_per_label,
        "speaker_transforms":    transforms,         # similarity (c, R, t)
        "speaker_transforms_rigid": {s: (1.0, R, t) for s, (R, t) in transforms_rigid.items()},
        "speaker_means":         speaker_means,
        "speaker_means_aligned": speaker_means_aligned,
        "all_contours_aligned":  all_contours_aligned,
        "contours_resampled":    contours_resampled,
        "common_labels":         common_labels,
        "nc_per_label":          nc,
    }
    return result


print("compute_mean_speaker() v2 defined ✓  (rigid GPA, pixel-scale preserving)")


In [ ]:

# =====================================================================
#  Run GPA on all loaded speakers
# =====================================================================

# Build input dict: { speaker_id: { label: pts (variable_nc, 2) } }
contours_by_speaker = {
    key: spk["contours"]
    for key, spk in speakers.items()
}

# Run mean-speaker computation
ms = compute_mean_speaker(contours_by_speaker, nc_resample=50,
                          max_iter=100, tol=1e-8, verbose=True)

# ── Per-speaker summary ──────────────────────────────────────────────────
print(f"\n{'=' * 75}")
print(f"{'SPEAKER':30s} {'scale':>7s} {'rot(°)':>8s} {'tx':>8s} {'ty':>8s}  "
      f"{'RMS_bef':>8s} {'RMS_aft':>8s}")
print(f"{'=' * 75}")

template = ms["template_mean"]

for s in sorted(ms["speaker_transforms"]):
    c, R, t = ms["speaker_transforms"][s]
    angle = np.degrees(np.arctan2(R[1, 0], R[0, 0]))
    raw   = ms["speaker_means"][s]
    alig  = ms["speaker_means_aligned"][s]

    # RMS before: raw vs initial centroid (not meaningful without centering,
    # so compare to template for consistency)
    rms_before = np.sqrt(np.mean((raw - raw.mean(0) - (template - template.mean(0))) ** 2))
    rms_after  = np.sqrt(np.mean((alig - template) ** 2))

    print(f"  {s:28s} {c:7.4f} {angle:8.2f} {t[0]:8.1f} {t[1]:8.1f}  "
          f"{rms_before:8.2f} {rms_after:8.2f}")

print(f"{'=' * 75}")
print(f"Template shape: {template.shape}")
print(f"Common labels:  {ms['common_labels']}")


In [ ]:

# =====================================================================
#  Visualization  (fixed — per-label template, shared axes)
# =====================================================================
common_labels = ms["common_labels"]
nc  = ms["nc_per_label"]
template = ms["template_mean"]
speaker_ids = sorted(ms["speaker_transforms"].keys())
cmap_lbl = plt.get_cmap("tab10")

# ── Fig A: Before vs After alignment ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(22, 10))

# (i) Before — raw speaker means, centred for visual comparison
ax = axes[0]
for si, s in enumerate(speaker_ids):
    raw = ms["speaker_means"][s]
    raw_c = raw - raw.mean(axis=0)
    # Draw each articulator segment separately (no cross-label lines)
    for li, lbl in enumerate(common_labels):
        seg = raw_c[li * nc : (li + 1) * nc]
        ax.plot(seg[:, 0], seg[:, 1], '-', color=plt.cm.tab10(si % 10),
                lw=0.8, alpha=0.5, label=s if li == 0 and si < 6 else None)
ax.set_title("Before alignment\n(raw speaker means, centred)", fontsize=14, fontweight='bold')
ax.set_aspect('equal'); ax.invert_yaxis(); ax.grid(True, alpha=0.3)
ax.legend(fontsize=7, loc='upper right', ncol=2)

# (ii) After — aligned speaker means + template (per-label segments)
ax = axes[1]
for si, s in enumerate(speaker_ids):
    alig = ms["speaker_means_aligned"][s]
    for li, lbl in enumerate(common_labels):
        seg = alig[li * nc : (li + 1) * nc]
        ax.plot(seg[:, 0], seg[:, 1], '-', color=plt.cm.tab10(si % 10),
                lw=0.8, alpha=0.4, label=s if li == 0 and si < 6 else None)
# Template — draw each articulator separately with its own colour
for li, lbl in enumerate(common_labels):
    seg = template[li * nc : (li + 1) * nc]
    ax.plot(seg[:, 0], seg[:, 1], '-', color='black', lw=3, alpha=0.85,
            label=f'template: {lbl}' if li < 4 else None)
ax.set_title("After GPA alignment  (pixel scale)\naligned means + template (black)",
             fontsize=14, fontweight='bold')
ax.set_aspect('equal'); ax.invert_yaxis(); ax.grid(True, alpha=0.3)
ax.legend(fontsize=7, loc='upper right', ncol=2)

plt.tight_layout()
plt.show()


# ── Fig B: Per-label aligned contours  (SHARED axis limits) ──────────────
# Compute global bounding box so all subplots share the same scale
all_pts = []
for s in speaker_ids:
    for lbl in common_labels:
        all_pts.append(ms["all_contours_aligned"][s][lbl])
all_pts = np.vstack(all_pts)
margin = 10
gx_min, gx_max = all_pts[:, 0].min() - margin, all_pts[:, 0].max() + margin
gy_min, gy_max = all_pts[:, 1].min() - margin, all_pts[:, 1].max() + margin

ncols_b = min(len(common_labels), 4)
nrows_b = (len(common_labels) + ncols_b - 1) // ncols_b
fig, axes = plt.subplots(nrows_b, ncols_b, figsize=(6 * ncols_b, 5 * nrows_b))
axes = np.array(axes).reshape(-1)

for li, lbl in enumerate(common_labels):
    ax = axes[li]
    for si, s in enumerate(speaker_ids):
        pts = ms["all_contours_aligned"][s][lbl]
        ax.plot(pts[:, 0], pts[:, 1], '-', color=plt.cm.tab10(si % 10),
                lw=1.0, alpha=0.6, label=s if li == 0 else None)
    # Template for this label
    tmpl_lbl = ms["template_per_label"][lbl]
    ax.plot(tmpl_lbl[:, 0], tmpl_lbl[:, 1], 'k-', lw=3, alpha=0.9)
    ax.set_title(lbl, fontsize=12, fontweight='bold')
    # Same scale for all subplots so shapes are comparable
    ax.set_xlim(gx_min, gx_max); ax.set_ylim(gy_max, gy_min)  # inverted y
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

for j in range(len(common_labels), len(axes)):
    axes[j].axis('off')

handles, labels_leg = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_leg, loc='lower center', ncol=5, fontsize=8,
           bbox_to_anchor=(0.5, -0.02))
fig.suptitle("Per-articulation aligned contours — shared scale\n"
             "black = template mean  (shapes preserve anatomical proportions)",
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()


# ── Fig C: Template mean speaker — all labels coloured ───────────────────
fig, ax = plt.subplots(figsize=(10, 10))
for li, lbl in enumerate(common_labels):
    tmpl_lbl = ms["template_per_label"][lbl]
    color = cmap_lbl(li % 10)
    ax.plot(tmpl_lbl[:, 0], tmpl_lbl[:, 1], '-o', color=color, lw=2.5, ms=3,
            label=lbl)
ax.set_title("Template Mean Speaker\n(all articulations — pixel scale)",
             fontsize=15, fontweight='bold')
ax.set_aspect('equal'); ax.invert_yaxis(); ax.grid(True, alpha=0.3)
ax.legend(fontsize=10, loc='upper right')
plt.tight_layout()
plt.show()


In [ ]:

# =====================================================================
#  MEDIAN SPEAKER — find the most representative speaker
# =====================================================================
# Three complementary metrics:
#   1. RMS to template : distance of aligned super-shape to the GPA mean
#   2. Sum-of-distances : total distance to ALL other speakers (geometric median)
#   3. Per-label RMS    : breakdown by articulator
#
# The speaker with the smallest "sum-of-distances to others" is the
# geometric-median speaker (most central among the population).

common_labels = ms["common_labels"]
nc  = ms["nc_per_label"]
template = ms["template_mean"]
speaker_ids = sorted(ms["speaker_transforms"].keys())

# ── 1. RMS to template ──────────────────────────────────────────────────
rms_to_template = {}
for s in speaker_ids:
    alig = ms["speaker_means_aligned"][s]
    rms_to_template[s] = np.sqrt(np.mean((alig - template) ** 2))

# ── 2. Pairwise distances → sum-of-distances (geometric median proxy) ───
pairwise = np.zeros((len(speaker_ids), len(speaker_ids)))
for i, si in enumerate(speaker_ids):
    for j, sj in enumerate(speaker_ids):
        if i != j:
            d = np.sqrt(np.mean(
                (ms["speaker_means_aligned"][si] - ms["speaker_means_aligned"][sj]) ** 2))
            pairwise[i, j] = d

sum_dist = {s: pairwise[i].sum() for i, s in enumerate(speaker_ids)}

# ── 3. Per-label RMS to template ────────────────────────────────────────
per_label_rms = {s: {} for s in speaker_ids}
for s in speaker_ids:
    for lbl in common_labels:
        pts = ms["all_contours_aligned"][s][lbl]
        tmpl = ms["template_per_label"][lbl]
        per_label_rms[s][lbl] = np.sqrt(np.mean((pts - tmpl) ** 2))

# ── Print ranked table ──────────────────────────────────────────────────
import pandas as pd

rows = []
for s in speaker_ids:
    row = {
        "Speaker": s,
        "RMS_to_template": rms_to_template[s],
        "Sum_dist_to_others": sum_dist[s],
    }
    for lbl in common_labels:
        row[lbl] = per_label_rms[s][lbl]
    rows.append(row)

df_median = pd.DataFrame(rows)
df_median = df_median.sort_values("Sum_dist_to_others").reset_index(drop=True)
df_median.index.name = "Rank"
df_median.index += 1   # 1-based rank

# Identify the median speaker
median_speaker = df_median.iloc[0]["Speaker"]
closest_to_mean = df_median.sort_values("RMS_to_template").iloc[0]["Speaker"]

print("=" * 90)
print("MEDIAN SPEAKER ANALYSIS  (ranked by sum-of-distances to all others)")
print("=" * 90)
# Format numeric columns
fmt_cols = [c for c in df_median.columns if c != "Speaker"]
df_show = df_median.copy()
for c in fmt_cols:
    df_show[c] = df_show[c].map(lambda v: f"{v:.2f}")
print(df_show.to_string())
print("=" * 90)
print(f"\n  ★ GEOMETRIC MEDIAN speaker (most central): {median_speaker}")
print(f"    (smallest total distance to all other speakers)")
print(f"\n  ★ CLOSEST TO MEAN speaker:                 {closest_to_mean}")
print(f"    (smallest RMS to the GPA template)")

# ── Visualization ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(28, 8))

# (A) Bar chart: Sum of distances (median metric)
ax = axes[0]
colors = ['#2ecc71' if s == median_speaker else '#3498db' for s in df_median["Speaker"]]
ax.barh(range(len(df_median)), df_median["Sum_dist_to_others"].values,
        color=colors, edgecolor='navy', height=0.6)
for i, (_, row) in enumerate(df_median.iterrows()):
    ax.text(row["Sum_dist_to_others"] + 1, i,
            f'{row["Sum_dist_to_others"]:.1f}', va='center', fontsize=9, fontweight='bold')
ax.set_yticks(range(len(df_median)))
ax.set_yticklabels(df_median["Speaker"], fontsize=9)
ax.set_xlabel("Sum of distances to all others (px)", fontsize=11)
ax.set_title("Geometric Median Ranking\n(green = median speaker)", fontsize=13, fontweight='bold')
ax.invert_yaxis(); ax.grid(True, axis='x', alpha=0.3)

# (B) Bar chart: RMS to template
ax = axes[1]
df_by_rms = df_median.sort_values("RMS_to_template")
colors2 = ['#e74c3c' if s == closest_to_mean else '#95a5a6' for s in df_by_rms["Speaker"]]
ax.barh(range(len(df_by_rms)), df_by_rms["RMS_to_template"].values,
        color=colors2, edgecolor='navy', height=0.6)
for i, (_, row) in enumerate(df_by_rms.iterrows()):
    ax.text(row["RMS_to_template"] + 0.3, i,
            f'{row["RMS_to_template"]:.1f}', va='center', fontsize=9, fontweight='bold')
ax.set_yticks(range(len(df_by_rms)))
ax.set_yticklabels(df_by_rms["Speaker"], fontsize=9)
ax.set_xlabel("RMS to GPA template (px)", fontsize=11)
ax.set_title("Closest to Mean Ranking\n(red = closest to template)", fontsize=13, fontweight='bold')
ax.invert_yaxis(); ax.grid(True, axis='x', alpha=0.3)

# (C) Overlay: median speaker vs template
ax = axes[2]
for li, lbl in enumerate(common_labels):
    tmpl = ms["template_per_label"][lbl]
    color = cmap_lbl(li % 10)
    ax.plot(tmpl[:, 0], tmpl[:, 1], '-', color=color, lw=3, alpha=0.4)
    pts = ms["all_contours_aligned"][median_speaker][lbl]
    ax.plot(pts[:, 0], pts[:, 1], '-', color=color, lw=2, alpha=0.9, label=lbl)
ax.set_title(f"Median speaker: {median_speaker}\nvs template (faded)",
             fontsize=13, fontweight='bold')
ax.set_aspect('equal'); ax.invert_yaxis(); ax.grid(True, alpha=0.3)
ax.legend(fontsize=8, loc='upper right')

plt.tight_layout()
plt.show()

# ── Pairwise distance heatmap ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(pairwise, cmap='YlOrRd', interpolation='nearest')
ax.set_xticks(range(len(speaker_ids)))
ax.set_xticklabels(speaker_ids, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(speaker_ids)))
ax.set_yticklabels(speaker_ids, fontsize=8)
for i in range(len(speaker_ids)):
    for j in range(len(speaker_ids)):
        ax.text(j, i, f'{pairwise[i,j]:.1f}', ha='center', va='center', fontsize=7,
                color='white' if pairwise[i,j] > pairwise.max()*0.6 else 'black')
plt.colorbar(im, ax=ax, label='RMS distance (px)')
ax.set_title("Pairwise RMS distances between aligned speakers", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
